# Work Rate Limited Variable Duration Pendulum Swing Up

The positive work done by the torque input is calculated using a soft plus
function to avoid non-smoothness that would arise from using a standard max.

In this example, the goal is to put a limit on the work that can be done in 2 seconds.


In [ ]:
import os
import numpy as np
import sympy as sm
from opty import Problem
import matplotlib.pyplot as plt
import matplotlib.animation as animation

### Start with defining the fixed duration and number of nodes.

In [ ]:
target_angle = np.pi
num_nodes = 501

Set up equations of motion \
The last equation is added so V(t) can be bound

In [ ]:
m, g, d, t, h = sm.symbols("m, g, d, t, h", real=True)
theta, omega, T, V, X = sm.symbols(
    "theta, omega, T, V, X", cls=sm.Function
)

state_symbols = (theta(t), omega(t))
constant_symbols = (m, g, d)
specified_symbols = (T(t), V(t), X(t))

eom = sm.Matrix(
    [
        theta(t).diff() - omega(t),
        m * d**2 * omega(t).diff() + m * g * d * sm.sin(theta(t)) - T(t),
        X(t) - V(t),
    ]
)
eom

Specify the known system parameters.

In [ ]:
par_map = {
    m: 1.0,
    g: 9.81,
    d: 1.0,
}
time_delay = 2.0  # seconds

Generate the known trajectories to limit the power V(t) which can be released 
in any $time_{delay}$ time span.


I assume: V(t) = $\int_{t - time_{delay}}^{t}(T(s) \cdot \omega(s) ds)$, with T(s), $\omega(s)$ = 0 for s $\leq 0$ \
Further assumption: $T(s) \cdot \omega(s) = 0 $ if $T(s) \cdot \omega(s) \leq 0$ 

So, $\dfrac{d}{dt} V(t)$ is calculated as below:


$\dfrac{d}{dt} V(t) = T(t) \cdot \omega(t) - T(t - time_{delay}) \cdot \omega(t - time_{delay})$

In [ ]:
def delay_traj(free):
    # determine time_delay in units of h
    delay_num = min(int(time_delay / free[-1]), num_nodes)
    E_np = np.empty(num_nodes)
    # Basically the definition of the Riemannian integral as a sum is applied.
    for i in range(num_nodes):
        summe = 0.0
        for j in range(i - delay_num, i):
            if j <= 0:
                summe += 0.0
            elif free[2*num_nodes + j] * free[num_nodes + j] <= 0:
                summe += 0.0
            else:
                summe += free[2*num_nodes + j] * free[num_nodes + j] * free[-1]
        E_np[i] = summe
    return E_np


def delaydt_traj(free):
    delay_num = min(int(time_delay / free[-1]), num_nodes)
    # Create an array holding T(n * h) *omega(n * h) for n = 0 ... num_nodes-1
    E_help = np.array([free[2*num_nodes + i] * free[num_nodes + i] if
                        free[num_nodes + i] * free[2*num_nodes + i] >= 0
                        else 0.0
                       for i in range(num_nodes)])

    E_dt_np = np.empty(num_nodes)
    for i in range(num_nodes):
        j = i - delay_num
        if j <= 0:
            E_dt_np[i] = E_help[i]
        else:
            E_dt_np[i] = E_help[i] - E_help[j]
    return E_dt_np


mimimize $\int_0^{tf} T(s)^2 ds $

In [ ]:
def obj(free):
    """Minimize the sum of the squares of the control torque."""
    summe = np.sum(free[2*num_nodes:3*num_nodes]**2) * free[-1]
    return summe

def obj_grad(free):
    grad = np.zeros_like(free)
    grad[2*num_nodes:3*num_nodes] = 2.0 * free[-1] * free[2*num_nodes:3*num_nodes]
    grad[-1] = np.sum(free[2*num_nodes:3*num_nodes]**2)
    return grad

Specify the symbolic instance constraints, i.e. initial and end conditions using node numbers 0 to N - 1

In [ ]:
instance_constraints = (
    theta(0 * h),
    theta((num_nodes - 1) * h) - target_angle,
    omega(0 * h),
    omega((num_nodes - 1) * h),
)

Specify the variable bounds for each state and input.

In [ ]:
bounds = {
    T(t): (-2.0, 2.0),
    h: (0.0, 0.5),
}

Get a reasonable initial guess \
Here the eon is without the bound on V(t)

In [ ]:
eom = sm.Matrix(
    [
        theta(t).diff() - omega(t),
        m * d**2 * omega(t).diff() + m * g * d * sm.sin(theta(t)) - T(t),
    ]
)

prob = Problem(
    obj,
    obj_grad,
    eom,
    state_symbols,
    num_nodes,
    h,
    known_parameter_map=par_map,
    instance_constraints=instance_constraints,
    time_symbol=t,
    bounds=bounds,
    backend="numpy",
)

initial_guess = np.full(prob.num_free, 1.e-5)
solution, info = prob.solve(initial_guess)
print(info["status_msg"])
print(info["obj_val"])
print(prob.num_free)

_ = prob.plot_trajectories(solution, show_bounds=True)

Use the solution from above, and add an initial guess for V(t)

In [ ]:
eom = sm.Matrix(
    [
        theta(t).diff() - omega(t),
        m * d**2 * omega(t).diff() + m * g * d * sm.sin(theta(t)) - T(t),
        X(t) - V(t),
    ]
)

s1 = list(solution[0:-1])
s2 = list(delay_traj(solution))
solution = np.array(s1 + s2 + [solution[-1]])

for steps in range(2):
    bounds = {
    T(t): (-2.0, 2.0),
    h: (0.0, 0.5),
    X(t): (0.0, 15.0 - steps),
    }

    prob = Problem(
        obj,
        obj_grad,
        eom,
        state_symbols,
        num_nodes,
        h,
        known_parameter_map=par_map,
        instance_constraints=instance_constraints,
        time_symbol=t,
        bounds=bounds,
        known_trajectory_map={
            V(t): delay_traj,
            V(t).diff(t): delaydt_traj,
        },
        tmp_dir='delay_run',
    )
    prob.add_option('max_iter', 5000)
    initial_guess = solution
    solution, info = prob.solve(initial_guess)
    print(info["status_msg"])
    print(info["obj_val"])

Plot the constraint violations.

In [ ]:
_ = prob.plot_constraint_violations(solution, subplots=True)

Plot trajectories

In [ ]:
_ = prob.plot_trajectories(solution, show_bounds=True)

Plot objective value

In [ ]:
_ = prob.plot_objective_value()